# Notebook 02 — Data Cleaning & Clinical Variable Dictionary

## Patient Similarity Network + Agent-Based Modelling for Readmission

### Project structure

```text
sna/
│
├── diabetes+130-us+hospitals+for+years+1999-2008/
│   ├── diabetic_data.csv
│   └── IDS_mapping.csv
│
├── notebooks/
│   ├── 01_dataset_audit.ipynb
│   └── 02_cleaning.ipynb
│
├── results/
│
└── figures/
```

### Purpose

This notebook prepares a **clean encounter-level table** and creates a
**clinical variable dictionary** before patient-level aggregation.

According to the project proposal, Notebook 02 must:

- handle `?` and missing values explicitly;
- identify identifier columns;
- standardize categorical variables;
- document which variables are used, excluded, or transformed;
- avoid using post-outcome information;
- produce a cleaned encounter table and a variable-selection table.

### Important methodological rule

This notebook does **not** yet aggregate encounters into patients, define the
final patient-level readmission target, build the similarity network, or train
a predictive model.

Those decisions belong to later notebooks.

The raw dataset is never overwritten.


## Notebook 01 dependency

Notebook 01 has already established that:

- the raw dataset loads successfully;
- `encounter_id`, `patient_nbr`, and `readmitted` are present;
- encounter IDs are unique;
- multiple encounters per patient exist;
- missing values and `?` values are present.

Therefore Notebook 02 starts from the raw dataset and performs controlled,
documented cleaning rather than silently modifying the original data.


In [2]:
# ============================================================
# Cell 1 — Imports
# ============================================================

import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd

print("Python version :", platform.python_version())
print("Pandas version :", pd.__version__)
print("NumPy version  :", np.__version__)


Python version : 3.11.9
Pandas version : 3.0.0
NumPy version  : 2.4.1


In [3]:
# ============================================================
# Cell 2 — Project paths
# ============================================================

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

DATASET_DIR = (
    PROJECT_ROOT /
    "diabetes+130-us+hospitals+for+years+1999-2008"
)

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = DATASET_DIR / "diabetic_data.csv"

if not DATA_FILE.exists():
    # Case-insensitive fallback
    candidates = [
        f for f in DATASET_DIR.iterdir()
        if f.is_file() and f.stem.lower() == "diabetic_data"
    ]

    if candidates:
        DATA_FILE = candidates[0]
    else:
        raise FileNotFoundError(
            f"Could not find diabetic_data.csv in {DATASET_DIR}"
        )

print("Project root      :", PROJECT_ROOT)
print("Dataset file      :", DATA_FILE)
print("Results directory :", RESULTS_DIR)
print("Figures directory :", FIGURES_DIR)


Project root      : c:\Users\Gayatri\OneDrive\Desktop\sna
Dataset file      : c:\Users\Gayatri\OneDrive\Desktop\sna\diabetes+130-us+hospitals+for+years+1999-2008\diabetic_data.csv
Results directory : c:\Users\Gayatri\OneDrive\Desktop\sna\results
Figures directory : c:\Users\Gayatri\OneDrive\Desktop\sna\figures


In [4]:
# ============================================================
# Cell 3 — Load raw data
# ============================================================

df_raw = pd.read_csv(DATA_FILE)

print("Raw dataset shape:", df_raw.shape)


Raw dataset shape: (101766, 50)


In [5]:
# ============================================================
# Cell 4 — Preserve baseline audit values
# ============================================================

baseline_rows = len(df_raw)
baseline_columns = len(df_raw.columns)
baseline_patients = df_raw["patient_nbr"].nunique()
baseline_encounters = df_raw["encounter_id"].nunique()

baseline_target_counts = (
    df_raw["readmitted"]
    .value_counts(dropna=False)
    .sort_index()
)

print("Baseline rows       :", baseline_rows)
print("Baseline columns    :", baseline_columns)
print("Baseline patients   :", baseline_patients)
print("Baseline encounters :", baseline_encounters)

print("\nBaseline target:")
print(baseline_target_counts)


Baseline rows       : 101766
Baseline columns    : 50
Baseline patients   : 71518
Baseline encounters : 101766

Baseline target:
readmitted
<30    11357
>30    35545
NO     54864
Name: count, dtype: int64


# 1. Identify identifiers and target

For this project:

- `patient_nbr` is the patient identifier and will be needed later for
  patient-level aggregation.
- `encounter_id` identifies an encounter.
- `readmitted` is the outcome variable.

Identifiers must **not** become similarity features. The target must also not
be used as an input feature for the similarity network.


In [6]:
# ============================================================
# Cell 5 — Define identifiers and target
# ============================================================

IDENTIFIER_COLUMNS = [
    "encounter_id",
    "patient_nbr"
]

TARGET_COLUMN = "readmitted"

assert all(col in df_raw.columns for col in IDENTIFIER_COLUMNS)
assert TARGET_COLUMN in df_raw.columns

print("Identifier columns:", IDENTIFIER_COLUMNS)
print("Target column     :", TARGET_COLUMN)


Identifier columns: ['encounter_id', 'patient_nbr']
Target column     : readmitted


# 2. Handle the `?` missing-value code

The raw dataset uses `?` to represent unknown/unavailable categorical values.

We convert **only the exact string `?`** to `pd.NA`.

We do not impute values in this notebook.

Why?

Because imputation is a modelling decision and should be performed only after
the feature set and train/validation/test strategy are established. This also
prevents us from pretending that an unknown value is a real clinical category.


In [7]:
# ============================================================
# Cell 6 — Count '?' before cleaning
# ============================================================

question_marks_before = int((df_raw == "?").sum().sum())

print("Total '?' cells before cleaning:", question_marks_before)


Total '?' cells before cleaning: 192849


In [8]:
# ============================================================
# Cell 7 — Convert exact '?' values to pd.NA
# ============================================================

df_clean = df_raw.copy()

# Only exact '?' values are converted.
df_clean = df_clean.replace("?", pd.NA)

question_marks_after = int((df_clean == "?").sum().sum())

print("Total '?' cells after cleaning:", question_marks_after)

if question_marks_after == 0:
    print("PASS — All exact '?' missing-value codes were handled.")
else:
    print("FAIL — '?' values remain.")


Total '?' cells after cleaning: 0
PASS — All exact '?' missing-value codes were handled.


In [9]:
# ============================================================
# Cell 8 — Missingness after '?' conversion
# ============================================================

missing_after = pd.DataFrame({
    "missing_count": df_clean.isna().sum(),
    "missing_percentage": (
        df_clean.isna().mean() * 100
    ).round(2)
}).sort_values(
    "missing_count",
    ascending=False
)

missing_after.head(30)


,missing_count,missing_percentage
weight,98569,96.86
max_glu_serum,96420,94.75
A1Cresult,84748,83.28
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02
patient_nbr,0,0.00


# 3. Standardize categorical/string variables

We standardize string formatting only:

- remove leading/trailing whitespace;
- preserve the underlying category meaning;
- keep missing values as missing.

We do **not** convert diagnosis codes or other clinical codes into arbitrary
numeric meanings. Their interpretation will be handled explicitly during
feature engineering.


In [10]:
# ============================================================
# Cell 9 — Standardize string columns
# ============================================================

string_columns = df_clean.select_dtypes(
    include=["object", "string"]
).columns.tolist()

for col in string_columns:
    df_clean[col] = df_clean[col].astype("string").str.strip()

print("Standardized string columns:", len(string_columns))


Standardized string columns: 37


In [11]:
# ============================================================
# Cell 10 — Check for blank strings

# Blank strings can occur after whitespace stripping.
blank_string_counts = {}

for col in string_columns:
    blank_string_counts[col] = int(
        (df_clean[col] == "").sum()
    )

blank_string_report = (
    pd.Series(blank_string_counts, name="blank_string_count")
    .sort_values(ascending=False)
)

blank_string_report[blank_string_report > 0]


Series([], Name: blank_string_count, dtype: int64)

In [12]:
# ============================================================
# Cell 11 — Convert blank strings to missing

for col in string_columns:
    df_clean[col] = df_clean[col].replace("", pd.NA)

remaining_blank_strings = sum(
    int((df_clean[col] == "").sum())
    for col in string_columns
)

print("Remaining blank strings:", remaining_blank_strings)

if remaining_blank_strings == 0:
    print("PASS — Blank strings are represented as missing.")
else:
    print("WARNING — Blank strings remain.")


Remaining blank strings: 0
PASS — Blank strings are represented as missing.


# 4. Initial variable-selection policy

The project needs a documented distinction between:

1. **identifier** — required for bookkeeping but never a model/similarity feature;
2. **target** — outcome, retained separately and never used as an input feature;
3. **candidate_feature** — potentially usable after later leakage review;
4. **excluded** — not used as a feature under the current project design;
5. **review_required** — may be usable depending on the prediction/intervention
   point that will be formally defined in Notebook 04.

### Important leakage principle

The proposal requires that post-outcome information must not be used.

Because Notebook 04 will formally define the prediction/intervention point,
variables whose temporal status depends on that definition are flagged for
review rather than silently assumed to be safe.


In [13]:
# ============================================================
# Cell 12 — Build initial variable-selection table
# ============================================================

variable_rows = []

# Variables that are definitely identifiers
identifier_set = set(IDENTIFIER_COLUMNS)

# Target is always separated from predictors
target_set = {TARGET_COLUMN}

# Variables that need explicit temporal review.
# These can be known at discharge, but their validity depends on the
# prediction/intervention point that will be defined in Notebook 04.
temporal_review_columns = {
    "discharge_disposition_id",
    "discharge_disposition",
    "time_in_hospital"
}

# Candidate clinical/context variables.
# The target and identifiers are excluded automatically.
for col in df_clean.columns:
    if col in identifier_set:
        role = "identifier"
        use_as_feature = False
        reason = "Required for record/patient identification; never a similarity feature."

    elif col in target_set:
        role = "target"
        use_as_feature = False
        reason = "Outcome variable; retained separately and excluded from feature inputs."

    elif col in temporal_review_columns:
        role = "review_required"
        use_as_feature = False
        reason = (
            "Temporal validity depends on the prediction/intervention point; "
            "review in Notebook 04 before use."
        )

    else:
        role = "candidate_feature"
        use_as_feature = True
        reason = "Candidate clinical/demographic/context variable; subject to later leakage review."

    variable_rows.append({
        "column": col,
        "dtype_after_cleaning": str(df_clean[col].dtype),
        "role": role,
        "use_as_feature_now": use_as_feature,
        "reason": reason
    })

variable_selection = pd.DataFrame(variable_rows)

variable_selection


,column,dtype_after_cleaning,role,use_as_feature_now,reason
0,encounter_id,int64,identifier,False,Required for record/patient identification; ne...
1,patient_nbr,int64,identifier,False,Required for record/patient identification; ne...
2,race,string,candidate_feature,True,Candidate clinical/demographic/context variabl...
3,gender,string,candidate_feature,True,Candidate clinical/demographic/context variabl...
4,age,string,candidate_feature,True,Candidate clinical/demographic/context variabl...
5,weight,string,candidate_feature,True,Candidate clinical/demographic/context variabl...
6,admission_type_id,int64,candidate_feature,True,Candidate clinical/demographic/context variabl...
7,discharge_disposition_id,int64,review_required,False,Temporal validity depends on the prediction/in...
8,admission_source_id,int64,candidate_feature,True,Candidate clinical/demographic/context variabl...
9,time_in_hospital,int64,review_required,False,Temporal validity depends on the prediction/in...


In [14]:
# ============================================================
# Cell 13 — Inspect variable groups
# ============================================================

print("IDENTIFIERS")
print(variable_selection.loc[
    variable_selection["role"] == "identifier",
    "column"
].tolist())

print("\nTARGET")
print(variable_selection.loc[
    variable_selection["role"] == "target",
    "column"
].tolist())

print("\nREVIEW REQUIRED")
print(variable_selection.loc[
    variable_selection["role"] == "review_required",
    "column"
].tolist())

print("\nCANDIDATE FEATURES:", (
    variable_selection["role"] == "candidate_feature"
).sum())


IDENTIFIERS
['encounter_id', 'patient_nbr']

TARGET
['readmitted']

REVIEW REQUIRED
['discharge_disposition_id', 'time_in_hospital']

CANDIDATE FEATURES: 45


# 5. Check high-missingness variables

Notebook 01 showed that some variables have extremely high missingness,
especially `weight`, `payer_code`, and `medical_specialty`.

We **do not automatically delete them here**.

Instead, we create a missingness review table. A final feature-exclusion
decision should be justified and recorded rather than based on an arbitrary
threshold hidden inside code.


In [15]:
# ============================================================
# Cell 14 — Missingness review for candidate variables
# ============================================================

missingness_review = variable_selection[
    ["column", "role"]
].copy()

missingness_review["missing_count"] = [
    df_clean[col].isna().sum()
    for col in missingness_review["column"]
]

missingness_review["missing_percentage"] = [
    round(df_clean[col].isna().mean() * 100, 2)
    for col in missingness_review["column"]
]

missingness_review = missingness_review.sort_values(
    "missing_percentage",
    ascending=False
)

missingness_review.head(30)


,column,role,missing_count,missing_percentage
5,weight,candidate_feature,98569,96.86
22,max_glu_serum,candidate_feature,96420,94.75
23,A1Cresult,candidate_feature,84748,83.28
11,medical_specialty,candidate_feature,49949,49.08
10,payer_code,candidate_feature,40256,39.56
2,race,candidate_feature,2273,2.23
20,diag_3,candidate_feature,1423,1.40
19,diag_2,candidate_feature,358,0.35
18,diag_1,candidate_feature,21,0.02
1,patient_nbr,identifier,0,0.00


In [16]:
# ============================================================
# Cell 15 — Flag extremely sparse variables for manual review
# ============================================================

HIGH_MISSINGNESS_THRESHOLD = 90.0

high_missing_columns = missingness_review.loc[
    missingness_review["missing_percentage"] >= HIGH_MISSINGNESS_THRESHOLD,
    "column"
].tolist()

print(
    f"Variables with >= {HIGH_MISSINGNESS_THRESHOLD:.0f}% missingness:"
)

for col in high_missing_columns:
    print("-", col)


Variables with >= 90% missingness:
- weight
- max_glu_serum


## Interpretation of the high-missingness flag

This is a **review flag**, not an automatic deletion rule.

For example, `weight` is extremely sparse. We should not fill almost all
missing weights with a fabricated value simply to make the column complete.

The final decision will be documented in this notebook after inspection.


In [17]:
# ============================================================
# Cell 16 — Record explicit exclusion of very sparse variables
# ============================================================

# Current project decision:
# Do not use variables with >=90% missingness as candidate features.
# They remain in the cleaned encounter table for traceability.

for col in high_missing_columns:
    mask = variable_selection["column"] == col
    variable_selection.loc[mask, "role"] = "excluded"
    variable_selection.loc[mask, "use_as_feature_now"] = False
    variable_selection.loc[mask, "reason"] = (
        f"Excluded from candidate features because "
        f"missingness is >= {HIGH_MISSINGNESS_THRESHOLD:.0f}%; "
        "retained in cleaned encounter table for traceability."
    )

print("Updated variable-selection decisions.")


Updated variable-selection decisions.


In [18]:
# ============================================================
# Cell 17 — Review final variable-selection table
# ============================================================

variable_selection


,column,dtype_after_cleaning,role,use_as_feature_now,reason
0,encounter_id,int64,identifier,False,Required for record/patient identification; ne...
1,patient_nbr,int64,identifier,False,Required for record/patient identification; ne...
2,race,string,candidate_feature,True,Candidate clinical/demographic/context variabl...
3,gender,string,candidate_feature,True,Candidate clinical/demographic/context variabl...
4,age,string,candidate_feature,True,Candidate clinical/demographic/context variabl...
5,weight,string,excluded,False,Excluded from candidate features because missi...
6,admission_type_id,int64,candidate_feature,True,Candidate clinical/demographic/context variabl...
7,discharge_disposition_id,int64,review_required,False,Temporal validity depends on the prediction/in...
8,admission_source_id,int64,candidate_feature,True,Candidate clinical/demographic/context variabl...
9,time_in_hospital,int64,review_required,False,Temporal validity depends on the prediction/in...


# 6. Protect identifiers and target from feature selection

The following must never enter the similarity representation:

```text
encounter_id
patient_nbr
readmitted
```

`patient_nbr` will be used later to aggregate encounters into one patient-level
representation, but it is not a clinical similarity feature.

The readmission outcome must also not define the similarity network.


In [19]:
# ============================================================
# Cell 18 — Feature safety checks
# ============================================================

feature_candidates = variable_selection.loc[
    variable_selection["use_as_feature_now"] == True,
    "column"
].tolist()

for forbidden in IDENTIFIER_COLUMNS + [TARGET_COLUMN]:
    assert forbidden not in feature_candidates, (
        f"LEAKAGE/IDENTIFIER ERROR: {forbidden} "
        "was included as a feature."
    )

print("PASS — Identifiers and target are excluded from current feature candidates.")
print("Candidate feature count:", len(feature_candidates))


PASS — Identifiers and target are excluded from current feature candidates.
Candidate feature count: 43


# 7. Inspect the cleaned categorical values

Before moving forward, verify that standardization did not unexpectedly
change category meanings.


In [20]:
# ============================================================
# Cell 19 — Important categorical value inspection
# ============================================================

important_categorical = [
    "race",
    "gender",
    "age",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "medical_specialty",
    "payer_code",
    "change",
    "diabetesMed",
    "readmitted"
]

for col in important_categorical:
    if col in df_clean.columns:
        print("\n" + "=" * 65)
        print(col)
        print("=" * 65)
        print(
            df_clean[col]
            .value_counts(dropna=False)
            .head(20)
        )



race
race
Caucasian          76099
AfricanAmerican    19210
<NA>                2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64[pyarrow]

gender
gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64[pyarrow]

age
age
[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)        161
Name: count, dtype: int64[pyarrow]

admission_type_id
admission_type_id
1    53990
3    18869
2    18480
6     5291
5     4785
8      320
7       21
4       10
Name: count, dtype: int64

discharge_disposition_id
discharge_disposition_id
1     60234
3     13954
6     12902
18     3691
2      2128
22     1993
11     1642
5      1184
25      989
4       815
7       623
23      412
13      399
14      372
28      139
8       108
15       63
24       48
9        21
17       14
Name: count, 

# 8. Preserve target separately

The target remains in the cleaned encounter table for traceability, but it is
explicitly separated from the feature list.

We are **not** converting `<30`, `>30`, and `NO` into a final binary patient
target here. Notebook 04 will define the readmission target and its prediction
point.


In [21]:
# ============================================================
# Cell 20 — Create explicit target table
# ============================================================

target_table = df_clean[
    ["encounter_id", "patient_nbr", TARGET_COLUMN]
].copy()

target_table.head()


,encounter_id,patient_nbr,readmitted
0,2278392,8222157,NO
1,149190,55629189,>30
2,64410,86047875,NO
3,500364,82442376,NO
4,16680,42519267,NO


In [22]:
# ============================================================
# Cell 21 — Verify target distribution was not changed
# ============================================================

clean_target_counts = (
    df_clean[TARGET_COLUMN]
    .value_counts(dropna=False)
    .sort_index()
)

comparison = pd.DataFrame({
    "raw_count": baseline_target_counts,
    "clean_count": clean_target_counts
}).fillna(0).astype(int)

comparison["difference"] = (
    comparison["clean_count"] - comparison["raw_count"]
)

comparison


,raw_count,clean_count,difference
readmitted,,,
<30,11357,11357,0
>30,35545,35545,0
NO,54864,54864,0


# 9. Save cleaned encounter-level data

The raw file is **never overwritten**.

The cleaned table is saved in `results/` so later notebooks can use the exact
same cleaned data rather than silently regenerating it.


In [23]:
# ============================================================
# Cell 22 — Save cleaned encounter table
# ============================================================

CLEANED_FILE = RESULTS_DIR / "02_cleaned_encounters.csv"

df_clean.to_csv(
    CLEANED_FILE,
    index=False
)

print("Saved:", CLEANED_FILE)


Saved: c:\Users\Gayatri\OneDrive\Desktop\sna\results\02_cleaned_encounters.csv


In [24]:
# ============================================================
# Cell 23 — Save variable-selection table
# ============================================================

VARIABLE_SELECTION_FILE = RESULTS_DIR / "02_variable_selection.csv"

variable_selection.to_csv(
    VARIABLE_SELECTION_FILE,
    index=False
)

print("Saved:", VARIABLE_SELECTION_FILE)


Saved: c:\Users\Gayatri\OneDrive\Desktop\sna\results\02_variable_selection.csv


In [25]:
# ============================================================
# Cell 24 — Save missingness review
# ============================================================

MISSINGNESS_FILE = RESULTS_DIR / "02_missingness_review.csv"

missingness_review.to_csv(
    MISSINGNESS_FILE,
    index=False
)

print("Saved:", MISSINGNESS_FILE)


Saved: c:\Users\Gayatri\OneDrive\Desktop\sna\results\02_missingness_review.csv


In [26]:
# ============================================================
# Cell 25 — Save explicit target table
# ============================================================

TARGET_FILE = RESULTS_DIR / "02_encounter_targets.csv"

target_table.to_csv(
    TARGET_FILE,
    index=False
)

print("Saved:", TARGET_FILE)


Saved: c:\Users\Gayatri\OneDrive\Desktop\sna\results\02_encounter_targets.csv


# 10. Final Notebook 02 checkpoint

The checkpoint verifies:

1. Row count has not unexpectedly changed.
2. Column count has not unexpectedly changed.
3. Patient count is unchanged.
4. Encounter IDs remain unique.
5. `?` values are gone.
6. Required identifiers remain.
7. Target remains available.
8. Target counts have not changed.
9. Identifiers and target are excluded from feature candidates.
10. A variable-selection table exists.
11. The cleaned table exists.

A manual review is still required for the feature list and temporal/relevance
decisions before Notebook 03.


In [27]:
# ============================================================
# Cell 26 — FINAL AUTOMATED CHECKPOINT
# ============================================================

print("=" * 75)
print("NOTEBOOK 02 — FINAL CHECKPOINT")
print("=" * 75)

checks = {}

checks["row_count_preserved"] = (
    len(df_clean) == baseline_rows
)

checks["column_count_preserved"] = (
    len(df_clean.columns) == baseline_columns
)

checks["patient_count_preserved"] = (
    df_clean["patient_nbr"].nunique() == baseline_patients
)

checks["encounter_ids_unique"] = (
    df_clean["encounter_id"].nunique() == len(df_clean)
)

checks["no_question_marks"] = (
    (df_clean == "?").sum().sum() == 0
)

checks["identifiers_present"] = all(
    col in df_clean.columns
    for col in IDENTIFIER_COLUMNS
)

checks["target_present"] = (
    TARGET_COLUMN in df_clean.columns
)

checks["target_counts_preserved"] = (
    comparison["difference"].eq(0).all()
)

checks["identifiers_not_features"] = all(
    col not in feature_candidates
    for col in IDENTIFIER_COLUMNS
)

checks["target_not_feature"] = (
    TARGET_COLUMN not in feature_candidates
)

checks["variable_selection_complete"] = (
    len(variable_selection) == len(df_clean.columns)
)

checks["cleaned_file_exists"] = CLEANED_FILE.exists()

checks["variable_selection_file_exists"] = (
    VARIABLE_SELECTION_FILE.exists()
)

for name, result in checks.items():
    print(f"{'PASS' if result else 'FAIL':<6} | {name}")

print("=" * 75)

if all(checks.values()):
    print("OVERALL RESULT: PASS")
    print("Proceed to MANUAL REVIEW.")
else:
    print("OVERALL RESULT: FAIL")
    print("Fix the failed checks before moving to Notebook 03.")


NOTEBOOK 02 — FINAL CHECKPOINT
PASS   | row_count_preserved
PASS   | column_count_preserved
PASS   | patient_count_preserved
PASS   | encounter_ids_unique
PASS   | no_question_marks
PASS   | identifiers_present
PASS   | target_present
PASS   | target_counts_preserved
PASS   | identifiers_not_features
PASS   | target_not_feature
PASS   | variable_selection_complete
PASS   | cleaned_file_exists
PASS   | variable_selection_file_exists
OVERALL RESULT: PASS
Proceed to MANUAL REVIEW.


# Manual GO / STOP Review

## GO to Notebook 03 only if:

- [ ] Missing-value treatment is understood.
- [ ] `?` has been converted explicitly to missing.
- [ ] No arbitrary imputation was performed.
- [ ] Identifier columns are documented.
- [ ] `patient_nbr` is retained for patient aggregation but excluded as a feature.
- [ ] `encounter_id` is excluded as a feature.
- [ ] `readmitted` is retained as the outcome but excluded as a feature.
- [ ] High-missingness variables have explicit decisions.
- [ ] Candidate feature list has been manually reviewed.
- [ ] Variables requiring temporal review are documented.
- [ ] No post-outcome variable has been silently included.
- [ ] Cleaned encounter table was saved.
- [ ] Variable-selection table was saved.
- [ ] Final automated checkpoint says `OVERALL RESULT: PASS`.

## STOP if:

- Cleaning changes the number of encounter rows unexpectedly.
- Encounter IDs become duplicated.
- Patient IDs disappear.
- The target distribution changes unexpectedly.
- An identifier or target enters the feature set.
- A variable's temporal validity cannot be justified.
- Missing-value treatment is not reproducible.


In [28]:
# ============================================================
# Cell 27 — Compact summary for the project log
# ============================================================

summary = {
    "raw_rows": int(baseline_rows),
    "clean_rows": int(len(df_clean)),
    "raw_columns": int(baseline_columns),
    "clean_columns": int(len(df_clean.columns)),
    "raw_unique_patients": int(baseline_patients),
    "clean_unique_patients": int(df_clean["patient_nbr"].nunique()),
    "question_marks_before": int(question_marks_before),
    "question_marks_after": int(question_marks_after),
    "candidate_feature_count": int(len(feature_candidates)),
    "excluded_variable_count": int(
        (variable_selection["role"] == "excluded").sum()
    ),
    "review_required_count": int(
        (variable_selection["role"] == "review_required").sum()
    ),
    "identifier_count": int(
        (variable_selection["role"] == "identifier").sum()
    ),
    "target_count": int(
        (variable_selection["role"] == "target").sum()
    )
}

with open(
    RESULTS_DIR / "02_cleaning_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=4)

print(json.dumps(summary, indent=4))


{
    "raw_rows": 101766,
    "clean_rows": 101766,
    "raw_columns": 50,
    "clean_columns": 50,
    "raw_unique_patients": 71518,
    "clean_unique_patients": 71518,
    "question_marks_before": 192849,
    "question_marks_after": 0,
    "candidate_feature_count": 43,
    "excluded_variable_count": 2,
    "review_required_count": 2,
    "identifier_count": 2,
    "target_count": 1
}
